# Step 13 — Final human-validated ground truth (merge synthetic positives + 12c negatives)

> Build the **final** ground truth by folding, into the balanced Step 12b ground truth: (1) the **validated synthetic positives** (Step 12d, gated by the manual-evaluation CSV) and (2) the **12c negatives** (restoring the manually-judged absent labels 12b's balancing capped out). Deterministic — no LLM here.

**Paper section:** Step 3 — Ground-truth construction / Final ground truth.

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/12b_final_ground_truth.json` | JSON (balanced base) | Step 12b |
| `data/12d_synth_UnsafeDeployOnFork_present.json` | JSON (49 synthetic positives, frozen) | Step 12d |
| `data/12e_sinth_manual_evaluation.csv` | CSV (`11`-schema; approval gate) | Manual validation (A2/A3) |
| `data/12c_synthesis_targets_UnsafeDeployOnFork_absent.json` | JSON (50 negatives) | Step 12c |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/13_final_ground_truth.json` | JSON (12b schema) | agent-evaluation + analysis layer (UnsafeDeployOnFork) |
| `data/13_final_ground_truth.json.sha256.txt` | text | integrity |
| `data/step13_report.json` | JSON | audit (approval, LOC, fold counts, per-smell summary, hashes) |


In [1]:
# --- Notebook bootstrap ---
from dataset_utils.notebook_bootstrap import setup

setup()

import sys
from pathlib import Path

# Guarded __file__ hack: make dataset_utils importable when run as a plain script
# without an installed wheel. Harmless in Jupyter (__file__ undefined). Do NOT remove.
try:
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
except NameError:
    pass

from dataset_utils.synthetic_merge import SMELL_PASCAL, run

print("Step 13 - building final ground truth (12b + validated synth positives + 12c negatives)...", flush=True)

# Deterministic merge: approve (CSV) -> LOC pre-check -> normalize -> fold 12c
# negatives -> append positives -> write. Logic lives in dataset_utils.synthetic_merge.
output_path, report = run()

print("Final ground truth written.")
print(f"  Output : {output_path}")
print("  Report : data/step13_report.json")
print("-" * 48)
print(f"  Synth candidates      : {report['n_synth_candidates']}")
print(f"  Approved / rejected   : {report['n_approved']} / {report['n_rejected']}")
print(f"  Over-LOC excluded     : {report['n_over_loc']}")
print(f"  Positives merged      : {report['n_positives_merged']}")
fc = report["fold_counts"]
print(f"  12c negatives folded  : enriched {fc['enriched']}, added {fc['added']}, already {fc['already']}")
print(f"  Entries in -> out     : {report['n_entries_in']} -> {report['n_entries_out']}")
for r in report["smell_summary"]:
    if r["smell"] == SMELL_PASCAL:
        print(f"  {SMELL_PASCAL}: present {r['present']} ({r['present_synth']} synth) / "
              f"absent {r['absent']} ({r['absent_synth']} synth)")


Step 13 - building final ground truth (12b + validated synth positives + 12c negatives)...


Final ground truth written.
  Output : data/13_final_ground_truth.json
  Report : data/step13_report.json
------------------------------------------------
  Synth candidates      : 49
  Approved / rejected   : 45 / 4
  Over-LOC excluded     : 0
  Positives merged      : 45
  12c negatives folded  : enriched 40, added 5, already 5
  Entries in -> out     : 852 -> 902
  UnsafeDeployOnFork: present 50 (45 synth) / absent 50 (0 synth)


In [2]:
# --- Emit the manually-verified ground-truth coverage table (FINAL, with synth) ---
# Per-smell Pos/Neg counts + LOC range over the FINAL ground truth (13), grouped by
# smell type. Renders data/tables/ground_truth_validated_coverage.tex for the paper.
# (This table used to be emitted from Step 12b reading a downstream 12e artifact — a
# forward dependency. It now lives here, in the step that actually produces the final GT.)
import json
from pathlib import Path

from dataset_utils.latex import write_ground_truth_validated_coverage

FINAL_GROUND_TRUTH_PATH = Path("data/13_final_ground_truth.json")
COVERAGE_TEX = Path("data/tables/ground_truth_validated_coverage.tex")

with FINAL_GROUND_TRUTH_PATH.open(encoding="utf-8") as f:
    final_ground_truth = json.load(f)

write_ground_truth_validated_coverage(final_ground_truth, COVERAGE_TEX)
print(f"Coverage table -> {COVERAGE_TEX}  ({len(final_ground_truth)} entries)")


Coverage table -> data/tables/ground_truth_validated_coverage.tex  (902 entries)


In [3]:
# --- Emit the IRR tables from the FINAL (Step 12e) recalc ---
# The paper reports the detection-annotation agreement AFTER folding in the validated
# synthetic positives (Step 12e): mean kappa 0.8481 "Almost perfect", n=1818. The .tex
# used to be emitted by the deprecated _11b notebook, which read Step 11 (pre-synth,
# 0.7554 "Substantial") — so the replication package disagreed with the submitted paper.
# Emit here instead (Step 13 runs after 12e, so step12e_report.json already exists).
# Consumer: paper replication package. Uses relative paths like the coverage cell above
# (setup() has already chdir'd to the repo root; there is no ROOT in this notebook).
import json
from pathlib import Path

from dataset_utils.latex import write_irr_summary, write_irr_confusion_overall

STEP12E_REPORT = Path("data/step12e_report.json")
IRR_SUMMARY_TEX = Path("data/tables/irr_summary.tex")
IRR_CONFUSION_TEX = Path("data/tables/irr_confusion_overall.tex")

with STEP12E_REPORT.open(encoding="utf-8") as f:
    r12e = json.load(f)

write_irr_summary(r12e, IRR_SUMMARY_TEX)
write_irr_confusion_overall(r12e, IRR_CONFUSION_TEX)
print(f"IRR tables -> {IRR_SUMMARY_TEX}, {IRR_CONFUSION_TEX}")
print(f"  mean kappa {r12e['mean_kappa']} ({r12e['mean_kappa_interpretation']}), "
      f"n={r12e['overall']['n']}, confusion={r12e['overall']['confusion_matrix']}")


IRR tables -> data/tables/irr_summary.tex, data/tables/irr_confusion_overall.tex
  mean kappa 0.8481 (Almost perfect), n=1818, confusion={'pp': 794, 'pa': 37, 'ap': 111, 'aa': 876}


---

`13_final_ground_truth.json` is the human-validated ground truth for `UnsafeDeployOnFork` (50/50), consumed by the agent-evaluation module and the analysis layer.